In [6]:
import numpy as np, pandas as pd

BASE = 'csvs/'

In [7]:
def expected_score_no_labels(submission_csv,
                             test_csv=BASE + 'test.csv',
                             train_csv=BASE + 'train.csv',
                             slice_cols=('age_group', 'rurality', 'care_pathway'),
                             smooth_k=100):
    """Estimate the LB score of a test submission WITHOUT labels.
    Each test patient is assigned the TRAIN event rate of his/her slice
    (age_group x rurality x care_pathway), smoothed toward the global rate
    (Beta prior, k=100) so tiny cells don't produce extreme pseudo-rates.
    E[LL] = mean_i [ - r_i log p_i - (1-r_i) log(1-p_i) ].
    This is an ESTIMATE, not the leaderboard truth."""
    sub  = pd.read_csv(submission_csv)
    test = pd.read_csv(test_csv)
    trn  = pd.read_csv(train_csv)

    trn['age_group'] = pd.cut(trn['age'], bins=[0, 30, 50, 70, 120],
                              labels=['<30', '30-50', '50-70', '70+'])
    test['age_group'] = pd.cut(test['age'], bins=[0, 30, 50, 70, 120],
                               labels=['<30', '30-50', '50-70', '70+'])

    p0 = trn['readmitted_30d'].mean()
    g = trn.groupby(list(slice_cols), observed=True)['readmitted_30d'].agg(['sum', 'count'])
    g['r'] = (g['sum'] + smooth_k * p0) / (g['count'] + smooth_k)   # smoothed slice rate
    rates = g['r']

    m = test.merge(sub, on='patient_id', how='left')
    assert m['readmitted_30d'].notna().all(), "submission missing test IDs"
    r = m.set_index(list(slice_cols)).index.map(rates)
    r = pd.Series(r, index=m.index).fillna(p0)   # unseen slice combo -> global rate
    p = np.clip(m['readmitted_30d'].values, EPS, 1 - EPS)

    exp_ll = -float(np.mean(r * np.log(p) + (1 - r) * np.log(1 - p)))
    print(f"file            : {submission_csv}")
    print(f"pred mean/min/max: {p.mean():.4f} / {p.min():.4f} / {p.max():.4f}")
    print(f"EXPECTED log loss (slice-rate assumption): {exp_ll:.6f}")
    return exp_ll


In [ ]:
PATH = 'outputs/submission_01_constant.csv'   # <- change per submission
expected_score_no_labels(PATH)

file            : outputs/submission_01_constant.csv
pred mean/min/max: 0.1259 / 0.1259 / 0.1259
EXPECTED log loss (slice-rate assumption): 0.381996


0.381995829806123